# Make lines files

These are needed for the 'Directory Service'

Format: 
```
device_name   bmad_name eke_key s        z            path      area
SOLN:IN20:111 SOL1BK    SOLE    0.000000 2017.911482  CU_ALINE  GUN
CATH:IN20:111 CATHODE   INST    0.000000 2017.911482  CU_ALINE  GUN
SOLN:IN20:121 SOL1      SOLE    0.196010 2018.072044  CU_ALINE  GUN
QUAD:IN20:121 CQ01      MULT    0.196010 2018.072044  CU_ALINE  GUN
XCOR:IN20:121 XC00      HKIC    0.196010 2018.072044  CU_ALINE  GUN
```

In [1]:
from pytao import Tao
from pytao.util.parameters import tao_parameter_dict

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

In [2]:
# for testing
MODEL = 'sc_dasel'
TAO = Tao(f'-init $LCLS_LATTICE/bmad/models/{MODEL}/tao.init -noplot')

In [3]:
def get_areas(tao):
    begnames = tao.cmd('python lat_list 1@0>>BEG*|model ele.name')
    s = [float(x) for x in tao.cmd('python lat_list 1@0>>BEG*|model ele.s')][1:]
    areas = [n[3:] for n in begnames[1:]]
    # Add a BEGINNING area for s less than the first section
    areas.insert(0, 'BEGINNING')
    s.insert(0, s[0]-1e9)
    
    return areas, s

def area_at_s(s, areas, sbeg):
    """
    Returns the area name for an element at s
    
    """
    return areas[np.digitize(s, sbeg)-1]

AREAS, SBEG = get_areas(TAO)
for a, s in zip(AREAS, SBEG):
    print(a, s)

BEGINNING -1000000000.0
GUNB 0.0
L0B 2.052577
HTR 18.174867
COL0 42.2302745061275
L1B 83.4299445037505
BC1B 131.390924503751
COL1 140.21681107274
L2B 184.957792961131
BC2B 341.635532961132
EMIT2 368.133043107028
L3B 395.942353107028
EXT 653.638353107028
DOG 674.638353107028
BYP 1212.74946100023
SPD_1 2790.39416100023
SPD_2 2877.57016100023
DASEL 2921.37016100023
BSYA_2 3167.69440586055
B 3200.80027981768


In [4]:
%%tao
sho ele BEG*

-------------------------
Tao> sho ele BEG*
         0  BEGINNING                                        0.000
         1  BEGGUNB                                          0.000
        52  BEGL0B                                           2.053
       102  BEGHTR                                          18.175
       225  BEGCOL0                                         42.230
       325  BEGL1B                                          83.430
       454  BEGBC1B                                        131.391
       498  BEGCOL1                                        140.217
       599  BEGL2B                                         184.958
       989  BEGBC2B                                        341.636
      1027  BEGEMIT2                                       368.133
      1057  BEGL3B                                         395.942
      1695  BEGEXT                                         653.638
      1712  BEGDOG                                         674.638
      1884  BEGBYP

In [5]:
def get_floor(tao, elename):
    cmd = f'python ele:floor 1@0>>{elename}|model end'
    dat = tao_parameter_dict(tao.cmd(cmd))
    keys = 'floor_x floor_y floor_z floor_theta floor_phi floor_psi'.split()
    return dict(zip(keys, dat['Reference'].value))
get_floor(TAO, 'BEGINNING')

{'floor_x': 0.279996,
 'floor_y': 0.0,
 'floor_z': -10.044667,
 'floor_theta': 0.0,
 'floor_phi': 0.0,
 'floor_psi': 0.0}

In [6]:
def ele_head(tao, elename):
    cmd = f'python ele:head {elename}|model'
    param = tao_parameter_dict(tao.cmd(cmd))
    return {k:param[k].value for k in param}
    
ele_head(TAO, 123)

{'universe': 1,
 '1^ix_branch': 0,
 'ix_ele': 123,
 'key': 'Hkicker',
 'name': 'XC0H03',
 'type': 'class-6',
 'alias': 'XCOR:HTR:288',
 'descrip': '',
 'is_on': True,
 's': 23.233849,
 's_start': 23.233849,
 'ref_time': 7.84272898016141e-08,
 'has#methods': True,
 'has#ab_multipoles': True,
 'has#kt_multipoles': False,
 'has#multipoles_elec': True,
 'has#ac_kick': False,
 'has#taylor': False,
 'has#spin_taylor': False,
 'has#wake': False,
 'num#cartesian_map': 0,
 'num#cylindrical_map': 0,
 'num#taylor_field': 0,
 'num#grid_field': 0,
 'has#wall3d': 0,
 'has#control': False,
 'has#twiss': True,
 'has#mat6': True,
 'has#floor': True,
 'has#photon': False,
 'has#lord_slave': True}

In [7]:
AREAS[np.digitize(1719.7557465674, SBEG)-1]

'BYP'

In [8]:
def ele_table(tao, eles='-no_slaves 1@0>>*|model'):
    
    areas, area_sbeg =  get_areas(tao)

    ix_ele = tao.lat_list('*', 'ele.ix_ele', flags='-array_out -no_slaves')
    # ele head
    eles = [ele_head(tao, ix) for ix in ix_ele]
    # Add info
    for ele in eles:
        floor = get_floor(tao, ele['ix_ele'])
        ele.update(floor)
        ele['area'] = area_at_s(ele['s'], areas, area_sbeg)
    
    df = pd.DataFrame(eles)
    return df

df2 = ele_table(TAO)['name key s floor_z area alias'.split()]
df2 = df2[df2['alias']!=''].rename(columns={'alias':'DEVICE'})
df2['path'] = MODEL.upper()
df2

,name,key,s,floor_z,area,DEVICE,path
3,SOL1BKB,Solenoid,-0.071705,-10.116372,BEGINNING,SOLN:GUNB:100,SC_DASEL
5,CATHODEB,Instrument,0.000000,-10.044667,GUNB,CATH:GUNB:100,SC_DASEL
8,SOL1B,Solenoid,0.289580,-9.755087,GUNB,SOLN:GUNB:212,SC_DASEL
9,SQ01B,Multipole,0.246530,-9.798137,GUNB,QUAD:GUNB:212:2,SC_DASEL
10,CQ01B,Multipole,0.246530,-9.798137,GUNB,QUAD:GUNB:212:1,SC_DASEL
...,...,...,...,...,...,...,...
2036,XCSP1,Hkicker,2859.974461,2849.856303,SPD_1,XCOR:SPD:334,SC_DASEL
2038,BPMSP2,Monitor,2860.705061,2850.586903,SPD_1,BPMS:SPD:340,SC_DASEL
2040,QSP2,Quadrupole,2861.446161,2851.328003,SPD_1,QUAD:SPD:340,SC_DASEL
2042,YCSP2,Vkicker,2861.874661,2851.756503,SPD_1,YCOR:SPD:343,SC_DASEL


In [9]:
get_areas(TAO)[0]

['BEGINNING',
 'GUNB',
 'L0B',
 'HTR',
 'COL0',
 'L1B',
 'BC1B',
 'COL1',
 'L2B',
 'BC2B',
 'EMIT2',
 'L3B',
 'EXT',
 'DOG',
 'BYP',
 'SPD_1',
 'SPD_2',
 'DASEL',
 'BSYA_2',
 'B']

In [10]:
df2.to_csv('test_lines.dat', sep=' ', index=False)